In [1]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

# Load the data
data = pd.read_csv('E:\SignalModel\price 2024-01-01, 2024-08-27 min.csv')
data['datetime'] = pd.to_datetime(data['datetime'])
data.set_index('datetime', inplace=True)
data.sort_index(ascending=True,inplace=True)

In [2]:
def calculate_rsi(df, window):
    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.ewm(span=window,adjust=False).mean()
    avg_loss = loss.ewm(span=window, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    
    return rsi

# Zigzag Class
class Zigzag:
    def __init__(self, timeseries, swingthresh=None):
        self.timeseries = timeseries
        self.swingthresh = swingthresh

    def get_swings(self):
        values = self.timeseries['RSI']
        peak_indices, _ = find_peaks(values, prominence=values.max()*0.001, threshold=0.001)
        trough_indices, _ = find_peaks(-values, prominence=values.max()*0.001, threshold=0.001)

        extrema_indices = sorted(np.concatenate((peak_indices, trough_indices)))

        swings = []
        last_extremum_value = values.iloc[extrema_indices[0]]
        last_extremum_type = 'high' if extrema_indices[0] in peak_indices else 'low'

        swings.append((values.index[extrema_indices[0]], last_extremum_value, last_extremum_type))

        for idx in extrema_indices[1:]:
            value = values.iloc[idx]
            current_extremum_type = 'high' if idx in peak_indices else 'low'
            change = abs((value - last_extremum_value)) / last_extremum_value

            if current_extremum_type != last_extremum_type:
                if change >= self.swingthresh:
                    swings.append((values.index[idx], value, current_extremum_type))
                    last_extremum_value = value
                    last_extremum_type = current_extremum_type
            else:
                if (current_extremum_type == 'high' and value > last_extremum_value) or \
                   (current_extremum_type == 'low' and value < last_extremum_value):
                    swings[-1] = (values.index[idx], value, current_extremum_type)
                    last_extremum_value = value

        return swings

# Strategy Implementation
import pandas as pd
from datetime import timedelta
import logging

def rsi_divergence_strategy(data):
    logging.basicConfig(level=logging.DEBUG)
    
    swings = Zigzag(data, swingthresh=0.01).get_swings()
    results = []

    i = 0
    while i < len(swings):
        date, rsi_value, extrema_type = swings[i]
        date = pd.to_datetime(date)
        
        logging.debug(f"Processing swing: {i}, Date: {date}, RSI: {rsi_value}, Type: {extrema_type}")
        
        # Check for overbought condition and peak
        if rsi_value >= 70 and extrema_type == 'high':
            first_peak = (date, rsi_value, data['high'][date])
            logging.debug(f"First peak found: {first_peak}")
            reset_search = False
            for j in range(i + 1, len(swings)):
                next_date, next_rsi_value, next_extrema_type = swings[j]
                next_date = pd.to_datetime(next_date)

                logging.debug(f"Checking next swing for peak: {j}, Date: {next_date}, RSI: {next_rsi_value}, Type: {next_extrema_type}")
                logging.debug(f"Conditions: next_extrema_type == 'high' -> {next_extrema_type == 'high'}, next_rsi_value > 60 -> {next_rsi_value > 60}, next_rsi_value < first_peak[1] -> {next_rsi_value < first_peak[1]}, data['high'][next_date] > first_peak[2] -> {data['high'][next_date] > first_peak[2]}, next_date - first_peak[0] <= timedelta(hours=24) -> {next_date - first_peak[0] <= timedelta(hours=24)}")

                if next_rsi_value <= 50:
                    logging.debug(f"RSI 50 or less, resetting search for peak")
                    reset_search = True
                    i = j
                    break

                if next_extrema_type == 'high' and next_rsi_value > 60:
                    if next_rsi_value < first_peak[1] and data['high'][next_date] > first_peak[2]:
                        if next_date - first_peak[0] <= timedelta(hours=24):
                            results.append({
                                'First Peak Date': first_peak[0], 'First Peak RSI': first_peak[1], 'First Peak High': first_peak[2],
                                'Second Peak Date': next_date, 'Second Peak RSI': next_rsi_value, 'Second Peak High': data['high'][next_date]
                            })
                            logging.debug(f"Divergence found (peak): {results[-1]}")
                            i = j + 1
                            reset_search = False
                            break
                    else:
                        if next_rsi_value > first_peak[1]:
                            first_peak = (next_date, next_rsi_value, data['high'][next_date])
                            logging.debug(f"Updated first peak: {first_peak}")
            if reset_search:
                continue
            else:
                i += 1

        # Check for oversold condition and trough
        elif rsi_value <= 30 and extrema_type == 'low':
            first_trough = (date, rsi_value, data['low'][date])
            logging.debug(f"First trough found: {first_trough}")
            reset_search = False
            for j in range(i + 1, len(swings)):
                next_date, next_rsi_value, next_extrema_type = swings[j]
                next_date = pd.to_datetime(next_date)

                logging.debug(f"Checking next swing for trough: {j}, Date: {next_date}, RSI: {next_rsi_value}, Type: {next_extrema_type}")
                logging.debug(f"Conditions: next_extrema_type == 'low' -> {next_extrema_type == 'low'}, next_rsi_value < 40 -> {next_rsi_value < 40}, next_rsi_value > first_trough[1] -> {next_rsi_value > first_trough[1]}, data['low'][next_date] < first_trough[2] -> {data['low'][next_date] < first_trough[2]}, next_date - first_trough[0] <= timedelta(hours=24) -> {next_date - first_trough[0] <= timedelta(hours=24)}")

                if next_rsi_value >= 50:
                    logging.debug(f"RSI 50 or more, resetting search for trough")
                    reset_search = True
                    i = j
                    break

                if next_extrema_type == 'low' and next_rsi_value < 40:
                    if next_rsi_value > first_trough[1] and data['low'][next_date] < first_trough[2]:
                        if next_date - first_trough[0] <= timedelta(hours=24):
                            results.append({
                                'First Trough Date': first_trough[0], 'First Trough RSI': first_trough[1], 'First Trough Low': first_trough[2],
                                'Second Trough Date': next_date, 'Second Trough RSI': next_rsi_value, 'Second Trough Low': data['low'][next_date]
                            })
                            logging.debug(f"Divergence found (trough): {results[-1]}")
                            i = j + 1
                            reset_search = False
                            break
                    else:
                        if next_rsi_value < first_trough[1]:
                            first_trough = (next_date, next_rsi_value, data['low'][next_date])
                            logging.debug(f"Updated first trough: {first_trough}")
            if reset_search:
                continue
            else:
                i += 1

        else:
            i += 1

    return pd.DataFrame(results)
# Calculate RSI for filtered data
data['RSI'] = calculate_rsi(data, window=14)

In [3]:
results = rsi_divergence_strategy(data)

DEBUG:root:Processing swing: 0, Date: 2024-01-01 03:00:00, RSI: 28.326510050365314, Type: low
DEBUG:root:First trough found: (Timestamp('2024-01-01 03:00:00'), 28.326510050365314, 42270.0)
DEBUG:root:Checking next swing for trough: 1, Date: 2024-01-01 04:00:00, RSI: 40.33677953326309, Type: high
DEBUG:root:Conditions: next_extrema_type == 'low' -> False, next_rsi_value < 40 -> False, next_rsi_value > first_trough[1] -> True, data['low'][next_date] < first_trough[2] -> True, next_date - first_trough[0] <= timedelta(hours=24) -> True
DEBUG:root:Checking next swing for trough: 2, Date: 2024-01-01 05:00:00, RSI: 27.352729847108378, Type: low
DEBUG:root:Conditions: next_extrema_type == 'low' -> True, next_rsi_value < 40 -> True, next_rsi_value > first_trough[1] -> False, data['low'][next_date] < first_trough[2] -> True, next_date - first_trough[0] <= timedelta(hours=24) -> True
DEBUG:root:Updated first trough: (Timestamp('2024-01-01 05:00:00'), 27.352729847108378, 42207.9)
DEBUG:root:Checki

In [6]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

# Create subplots
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=('Price', 'RSI'),
                    row_heights=[0.6, 0.4])

# Plot the candlestick chart for price
fig.add_trace(go.Candlestick(x=data.index,
                             open=data['open'],
                             high=data['high'],
                             low=data['low'],
                             close=data['close'],
                             name='Price'),
              row=1, col=1)

# Plot the RSI line
fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], mode='lines', name='RSI'),
              row=2, col=1)
fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)

# Plot the divergences
for _, divergence in results.iterrows():
    if 'First Peak Date' in divergence and pd.notna(divergence['First Peak Date']):
        color = 'red'
        fig.add_trace(go.Scatter(x=[divergence['First Peak Date'], divergence['Second Peak Date']], 
                                 y=[divergence['First Peak RSI'], divergence['Second Peak RSI']], 
                                 mode='lines', line=dict(color=color, width=2)),
                      row=2, col=1)
        fig.add_trace(go.Scatter(x=[divergence['First Peak Date'], divergence['Second Peak Date']], 
                                 y=[divergence['First Peak High'], divergence['Second Peak High']], 
                                 mode='lines', line=dict(color=color, width=2)),
                      row=1, col=1)
    if 'First Trough Date' in divergence and pd.notna(divergence['First Trough Date']):
        color = 'green'
        fig.add_trace(go.Scatter(x=[divergence['First Trough Date'], divergence['Second Trough Date']], 
                                 y=[divergence['First Trough RSI'], divergence['Second Trough RSI']], 
                                 mode='lines', line=dict(color=color, width=2)),
                      row=2, col=1)
        fig.add_trace(go.Scatter(x=[divergence['First Trough Date'], divergence['Second Trough Date']], 
                                 y=[divergence['First Trough Low'], divergence['Second Trough Low']], 
                                 mode='lines', line=dict(color=color, width=2)),
                      row=1, col=1)

# Customize layout
fig.update_layout(
    title='RSI and Price with Divergences',
    xaxis_title='Date',
    yaxis_title='Price',
    yaxis2_title='RSI',
    template='plotly_white',
    width=900,
    height=800,
    xaxis_rangeslider_visible=False
)

# Show plot
fig.show()